# 処理時間を計測する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/A_colab_python_tips/timing.ipynb)

NLP クックブック A-07 ／ Colab と Python の小技

どの処理が遅いのかを知りたい。2 つの実装のどちらが速いかを比べたい。

In [1]:
%%time
# セル全体の実行時間を測る（%%time はセルの 1 行目に書く）
total = sum(i * i for i in range(3_000_000))

CPU times: user 210 ms, sys: 0 ns, total: 210 ms
Wall time: 374 ms


In [2]:
import re

text = "自然言語処理 " * 10000
pattern = re.compile(r"\S+")

# %timeit: 何度も繰り返して平均と標準偏差を出す。短い処理の比較に向く
%timeit text.split()
%timeit pattern.findall(text)

595 µs ± 54.1 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
2.24 ms ± 258 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [3]:
import random
import time
from collections import Counter
from contextlib import contextmanager

@contextmanager
def timer(name):
    """with ブロックの時間を測る。スクリプトに書き写しても使える"""
    start = time.perf_counter()
    yield
    print(f"[{name}] {time.perf_counter() - start:.3f} 秒")

# 計測の例として、200 万語のテキストを作る
random.seed(0)
words = ["自然", "言語", "処理", "を", "学ぶ", "の", "は", "データ", "モデル", "文章", "単語", "解析"]
big_text = " ".join(random.choices(words, k=2_000_000))

with timer("トークン化"):
    tokens = big_text.split()
with timer("頻度の集計"):
    freq = Counter(tokens)

[トークン化] 0.210 秒
[頻度の集計] 0.203 秒


In [4]:
# どの関数に時間がかかっているかを調べる（プロファイラ）
import cProfile, pstats

def slow_pipeline():
    words = [w for w in text.split()]
    return sorted(set(words)), sum(len(w) for w in words), re.findall(r"処理", text)

profiler = cProfile.Profile(); profiler.runcall(slow_pipeline)
_ = pstats.Stats(profiler).sort_stats("cumulative").print_stats(5)

         20626 function calls (20620 primitive calls) in 0.012 seconds

   Ordered by: cumulative time
   List reduced from 143 to 5 due to restriction <5>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        4    0.000    0.000    0.010    0.003 /usr/lib/python3.13/asyncio/base_events.py:1981(_run_once)
    10001    0.006    0.000    0.007    0.000 /tmp/ipykernel_2366827/4064903067.py:6(<genexpr>)
        4    0.000    0.000    0.006    0.001 /usr/lib/python3.13/asyncio/events.py:87(_run)
        4    0.000    0.000    0.006    0.001 {method 'run' of '_contextvars.Context' objects}
        1    0.000    0.000    0.005    0.005 /usr/local/lib/python3.13/dist-packages/ipykernel/kernelbase.py:277(poll_control_queue)




- GPU の処理は非同期なので、正確に測るには計測の前後で `torch.cuda.synchronize()` を呼ぶ。
- 初回の実行は、モデルの読み込みやキャッシュの作成で遅い。2 回目以降を測る。
- Colab の無料枠は実行のたびに CPU や GPU の割り当てが変わる。異なるセッションで測った時間は比較できない。